# hh_mlp_whh（RNN 版 Markov MLP）を Colab の GPU で回す
ランタイムを GPU にして上から順に実行。Drive の `MyDrive/hh_whh_colab/` に `bundle.zip` を置いておく。
終わったジョブ（`out/<tag>.json` がある）は飛ばすので、切れたら実行のセルからやり直せばよい。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
D = '/content/drive/MyDrive/hh_whh_colab'   # bundle.zip を置いたフォルダ
R = '/content/repo'
os.makedirs(D + '/out/logs', exist_ok=True)
os.makedirs(D + '/cache', exist_ok=True)
subprocess.run(f'rm -rf {R} && mkdir -p {R} && cd {R} && unzip -q {D}/bundle.zip', shell=True, check=True)
# 採点の正解（HH の解）のキャッシュを Drive に置き、2 回目から計算を省く
for f in ('hh_bench_reference.npz', 'hh_bench_val.npz'):
    os.symlink(f'{D}/cache/{f}', f'{R}/results/{f}')
subprocess.run('pip -q install efel', shell=True)
subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True)
print('CPU コア数:', os.cpu_count())

In [ ]:
# 1 ステップずつ Python で回すので CPU（カーネルの起動）で律速される。Colab の CPU は 2 コアなので同時に 2 本まで
PARALLEL = 2
JOBS = '''# 閉ループ微調整（CPU では 1 回 200 秒で回せなかったもの）。系列 50 ms × 300 回、検証は 50 回ごと
ft --tag cft_r_mm --iters 300 --eval-every 50 --ep-ms 50 --log-every 10
ft --tag cft_m_mm --ctrl --iters 300 --eval-every 50 --ep-ms 50 --log-every 10
ft --tag cft_r_e01 --src results/hh_improve/sp_r_scr_e01.pt --iters 300 --eval-every 50 --ep-ms 50 --log-every 10
# 系列版の摂動事前学習の seed 違い（手元の seed 0 と同じ条件。ばらつきを見る）
pre --tag sp_r_scr_e01_s1 --seed 1 --init scratch --cell mlpr --eps0 0.1 --iters 30000 --lr 1e-3 --eval-every 5000
pre --tag sp_r_scr_e01_s2 --seed 2 --init scratch --cell mlpr --eps0 0.1 --iters 30000 --lr 1e-3 --eval-every 5000
pre --tag sp_r_init_s1 --seed 1 --init mm_rel --cell mlpr --eps0 0 --iters 20000 --lr 3e-4 --eval-every 5000
pre --tag sp_r_init_s2 --seed 2 --init mm_rel --cell mlpr --eps0 0 --iters 20000 --lr 3e-4 --eval-every 5000
'''
import shlex, sys, time
from concurrent.futures import ThreadPoolExecutor
lines = [l.strip() for l in JOBS.splitlines() if l.strip() and not l.startswith('#')]
tag_of = lambda l: shlex.split(l)[shlex.split(l).index('--tag') + 1]
# 正解のキャッシュを 1 つのプロセスで先に作る（同時に書かないように）
subprocess.run([sys.executable, '-X', 'utf8', '-c', "import sys; sys.path.insert(0, 'experiments/hh_surrogate'); "
                "import hh_bench as hb; hb.reference(); hb.reference(val=True)"], cwd=R, check=True)

def go(l):
    t = tag_of(l)
    if os.path.exists(f'{D}/out/{t}.json'):
        return t + ' (済)'
    t0 = time.time()
    with open(f'{D}/out/logs/{t}.log', 'w') as f:
        r = subprocess.run([sys.executable, '-u', '-X', 'utf8', 'experiments/hh_surrogate/hh_mlp_whh.py']
                           + shlex.split(l) + ['--device', 'cuda', '--out', f'{D}/out'],
                           cwd=R, stdout=f, stderr=subprocess.STDOUT)
    return f'{t} rc={r.returncode} {time.time() - t0:.0f} s'

print(len(lines), 'jobs,', PARALLEL, 'parallel', flush=True)
with ThreadPoolExecutor(PARALLEL) as ex:
    for s in ex.map(go, lines):
        print(s, flush=True)

In [ ]:
# 途中経過（ログの最後の数行）
import glob
for p in sorted(glob.glob(f'{D}/out/logs/*.log')):
    print('==', os.path.basename(p))
    print(''.join(open(p, encoding='utf-8', errors='replace').readlines()[-3:]))

In [ ]:
# 結果の一覧（テスト 26 本の全部一致した本数、Γ、検証で選んだ回）
import glob, json
for p in sorted(glob.glob(f'{D}/out/*.json')):
    d = json.load(open(p, encoding='utf-8'))
    if isinstance(d, dict) and 'test' in d:
        print(f"{os.path.basename(p):28s} {d['test']['n100']:2d}/26  Γ {d['test']['gamma']:.3f}  val best {d['best']}")